# Machine Learning Basics - Final Project
## Phase 2: Data Cleaning and Feature Extraction

**Course:** Machine Learning Basics (Fall 2025)
**Section:** Data Cleaning & Feature Extraction

---
### Notebook Description
This notebook covers the first two tasks of Phase 2:
1.  **Data Cleaning:** Preprocessing raw audio files (resampling, silence removal, normalization) and segmenting them into shorter clips to increase dataset size and consistency.
2.  **Feature Extraction:** Extracting time-domain and frequency-domain features (MFCCs, Spectral Centroid, ZCR, etc.) suitable for language identification.

In [11]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import warnings
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plot style
sns.set(style="whitegrid")

In [12]:
# --- Configuration & Constants ---

# Paths
RAW_DATA_PATH = './dataset'            # Path to the raw audio folders
PROCESSED_DATA_PATH = './processed_dataset'  # Path to save cleaned files
FEATURES_FILE = 'features.csv'         # Output CSV file

# Audio Parameters
TARGET_SR = 22050       # Sampling rate (Hz)
TOP_DB = 20             # Threshold for silence removal (dB)

# Target Languages (Classes)
LANGUAGES = ['German', 'Italian', 'Spanish', 'Korean']

## 1. Data Cleaning & Segmentation
In this step, we perform the following operations on the raw audio data:
1.  **Resampling:** Convert all audio to a consistent sampling rate (22.05 kHz).
2.  **Mono Conversion:** Ensure all channels are mono.
3.  **Silence Removal:** Trim silence from the beginning and end of the clips.
4.  **Normalization:** Normalize amplitude to the [-1, 1] range.
5.  **Segmentation:** Split long audio files (approx. 1 min) into 5-second overlapping chunks. This acts as a data augmentation technique.

In [13]:
# Cell 5 [Code]
def process_audio_file(file_path, save_dir, filename):
    """
    Reads an audio file, cleans it (Resample, Trim, Normalize), and saves it.
    """
    try:
        # Load audio (automatically converts to mono)
        y, sr = librosa.load(file_path, sr=TARGET_SR, mono=True)

        # 1. Remove Silence
        y_trimmed, _ = librosa.effects.trim(y, top_db=TOP_DB)
        
        # Skip if file is too short (e.g., empty or just noise)
        if len(y_trimmed) < TARGET_SR:
            return

        # 2. Normalization (Max Amplitude Scaling)
        max_val = np.max(np.abs(y_trimmed))
        if max_val > 0:
            y_norm = y_trimmed / max_val
        else:
            y_norm = y_trimmed

        # 3. Save the processed file (Without Chunking)
        output_path = os.path.join(save_dir, filename)
        sf.write(output_path, y_norm, TARGET_SR)

    except Exception as e:
        print(f"Error processing {filename}: {e}")

# --- Execution of Data Cleaning ---
print("Starting Data Cleaning (No Chunking)...")

for lang in LANGUAGES:
    input_folder = os.path.join(RAW_DATA_PATH, lang)
    output_folder = os.path.join(PROCESSED_DATA_PATH, lang)
    
    # Create output directory if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        
    if not os.path.exists(input_folder):
        print(f"Warning: Input folder for {lang} not found.")
        continue
        
    # Get list of audio files
    files = [f for f in os.listdir(input_folder) if f.lower().endswith(('.mp3', '.wav', '.flac'))]
    
    print(f"Processing {lang} ({len(files)} files)...")
    
    for f in tqdm(files, desc=f"{lang}"):
        process_audio_file(os.path.join(input_folder, f), output_folder, f)

print(f"\nPreprocessing complete. Data saved to '{PROCESSED_DATA_PATH}'.")

Starting Data Cleaning (No Chunking)...
Processing German (180 files)...


German: 100%|██████████| 180/180 [00:46<00:00,  3.83it/s]


Processing Italian (180 files)...


Italian: 100%|██████████| 180/180 [00:49<00:00,  3.63it/s]


Processing Spanish (180 files)...


Spanish: 100%|██████████| 180/180 [00:45<00:00,  3.93it/s]


Processing Korean (180 files)...


Korean: 100%|██████████| 180/180 [00:44<00:00,  4.00it/s]


Preprocessing complete. Data saved to './processed_dataset'.


## 2. Feature Extraction
We extract a set of statistical features from each audio chunk to represent its characteristics.
Since the classifiers require fixed-size input vectors, we calculate the **Mean** and **Standard Deviation** for each feature over time.

**Selected Features:**
* **MFCCs (1-13):** Capture the spectral envelope and timbral characteristics (crucial for phoneme distinction).
* **Spectral Centroid:** Represents the "brightness" of the sound.
* **Spectral Bandwidth:** Width of the spectral band.
* **Spectral Rolloff:** Frequency below which a certain percentage of total energy lies.
* **Zero Crossing Rate (ZCR):** Rate of sign-changes (useful for distinguishing voiced/unvoiced sounds).
* **RMS Energy:** Loudness/Amplitude of the signal.

In [15]:
def extract_features(file_path):
    """
    Extracts time and frequency domain features from a .wav file.
    Returns a dictionary of features.
    """
    try:
        y, sr = librosa.load(file_path, sr=None) # sr is already consistent

        features = {}

        # 1. MFCCs (13 coefficients)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        for i in range(mfcc.shape[0]):
            features[f'mfcc_mean_{i+1}'] = np.mean(mfcc[i])
            features[f'mfcc_std_{i+1}'] = np.std(mfcc[i])

        # 2. Spectral Centroid
        cent = librosa.feature.spectral_centroid(y=y, sr=sr)
        features['spec_cent_mean'] = np.mean(cent)
        features['spec_cent_std'] = np.std(cent)

        # 3. Spectral Bandwidth
        bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
        features['spec_bw_mean'] = np.mean(bw)
        features['spec_bw_std'] = np.std(bw)

        # 4. Spectral Rolloff
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        features['rolloff_mean'] = np.mean(rolloff)
        features['rolloff_std'] = np.std(rolloff)

        # 5. Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)
        features['zcr_mean'] = np.mean(zcr)
        features['zcr_std'] = np.std(zcr)

        # 6. RMS Energy
        rms = librosa.feature.rms(y=y)
        features['rms_mean'] = np.mean(rms)
        features['rms_std'] = np.std(rms)

        return features

    except Exception as e:
        print(f"Error extracting features from {file_path}: {e}")
        return None

# --- Execution of Feature Extraction ---
print("Starting Feature Extraction...")
data_records = []

for lang in LANGUAGES:
    lang_dir = os.path.join(PROCESSED_DATA_PATH, lang)
    
    if not os.path.exists(lang_dir):
        print(f"Skipping {lang} (Folder not found)")
        continue
        
    files = [f for f in os.listdir(lang_dir) if f.endswith('.mp3')]
    print(f"Extracting features for {lang} ({len(files)} samples)...")
    
    for f in tqdm(files, desc=f"{lang}"):
        f_path = os.path.join(lang_dir, f)
        feats = extract_features(f_path)
        
        if feats:
            feats['label'] = lang
            feats['filename'] = f
            data_records.append(feats)

# Save to CSV
if data_records:
    df = pd.DataFrame(data_records)
    # Move 'label' to the last column
    cols = [c for c in df.columns if c not in ['label', 'filename']]
    df = df[cols + ['label']]
    
    df.to_csv(FEATURES_FILE, index=False)
    print(f"\nFeature extraction complete. Data saved to '{FEATURES_FILE}'")
    print(f"Dataset Shape: {df.shape}")
else:
    print("No features extracted. Check paths.")

Starting Feature Extraction...
Extracting features for German (172 samples)...


German: 100%|██████████| 172/172 [01:00<00:00,  2.86it/s]


Extracting features for Italian (180 samples)...


Italian: 100%|██████████| 180/180 [01:07<00:00,  2.68it/s]


Extracting features for Spanish (180 samples)...


Spanish: 100%|██████████| 180/180 [00:58<00:00,  3.09it/s]


Extracting features for Korean (180 samples)...


Korean: 100%|██████████| 180/180 [00:58<00:00,  3.08it/s]


Feature extraction complete. Data saved to 'features.csv'
Dataset Shape: (712, 37)
